# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks

Before creating the baseline score, I tested two signals to evaluate whether the patterns in the data support potential action signals.

---

### Signal 1: Staleness / Refresh Signal (`days_since_last_update`)

**Hypothesis:** Pages that have not been updated for a longer duration (e.g., 91+ days) exhibit higher observed decline rates compared to fresher pages (0–30 days), providing descriptive support for a content refresh flag.

See the bucket table output in the code cell below.

**Verdict: MIXED**

*Explanation:* Pages unupdated for 91–180 days show an observed decline rate of **61.11%**, compared to **51.14%** for pages updated within 0–30 days (+9.97 percentage points higher decline rate for stale pages). Overall, pages with 91+ days since update have a 60.85% decline rate vs 51.20% for fresher pages (<91 days). However, the oldest bucket (181+ days) has a lower decline rate (47.13%) likely due to a smaller sample size (n=174) or evergreen content. Thus, the verdict is **MIXED**: while staleness does not guarantee decline, 91+ days provides a useful empirical threshold for human review.

---

### Signal 2: Search Position → Click-Through Rate (`avg_position` vs Weighted CTR)

**Hypothesis:** Pages ranking worse in Google search results (higher `avg_position`) receive lower overall click-through rates, supporting FlyRank's CTR-fix and search position logic.

See the bucket table output in the code cell below.

**Verdict: CONFIRMED**

*Explanation:* Using the ML-06 weighted CTR methodology (`total_clicks / total_impressions * 100`), weighted CTR decreases as search position worsens: **0.49%** for Top 3 (1–3), **0.35%** for Page 1 (3.1–10), **0.35%** for Striking Distance (10.1–20), **0.15%** for Pages 3–5 (20.1–50), and **0.04%** for Deep positions (50+). Pages with `avg_position == 0` (1,205 rows) represent missing position data rather than rank zero and are explicitly categorized as "no position data (0)". The weighted CTR pattern confirms that search rank visibility correlates directly with user click behavior.

---

### Section 1 Takeaway

Both signals provide useful descriptive evidence for decision-support:
1. **Staleness (`days_since_last_update >= 91`)** serves as a valid condition for identifying pages that may benefit from a content refresh review.
2. **Search Position / Weighted CTR** complements staleness by demonstrating that worse position rankings directly correspond to lower aggregate CTR.

*Note:* Neither signal is causal or a guaranteed predictor of Google's algorithm. They serve as transparent features for human decision-support.

In [ ]:
import pandas as pd
from pathlib import Path

# Load dataset from repository root
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), "Starter CSV not found — run this notebook from repository root."
df = pd.read_csv(DATA_PATH)

# Derive observed decline label from trend_direction for descriptive audit only
# (This label is never used as an input feature in the baseline score)
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

print("==================================================")
print("SIGNAL 1: Staleness / Refresh Signal")
print("==================================================")

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

# Calculate page count (n), decline count, and decline rate (%)
signal1_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_count=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
    .rename(columns={"staleness_bucket": "bucket"})
)
signal1_table["decline_rate"] = (signal1_table["decline_rate"] * 100).round(2)

print("\nSignal 1 Bucket Table (Staleness vs Observed Decline Rate):")
print(signal1_table.to_string(index=False))
print("\nVerdict: MIXED — 91-180 days shows elevated decline (61.11% vs 51.14% for 0-30 days), providing descriptive support for a refresh flag.")

print("\n==================================================")
print("SIGNAL 2: Search Position vs Weighted CTR")
print("==================================================")

# Create position buckets and handle avg_position == 0 explicitly
pos_df = df.copy()
pos_df["position_bucket"] = pd.cut(
    pos_df["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["top 3 (1-3)", "page 1 (3.1-10)", "striking distance (10.1-20)", "pages 3-5 (20.1-50)", "deep (50+)"],
    include_lowest=False
)
pos_df["position_bucket"] = pos_df["position_bucket"].cat.add_categories(["no position data (0)"])
pos_df.loc[pos_df["avg_position"] == 0, "position_bucket"] = "no position data (0)"

# Calculate page count (n), total_clicks, total_impressions, and weighted_ctr (%)
# weighted_ctr = (total clicks / total impressions) * 100
signal2_table = (
    pos_df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        total_clicks=("clicks_90d", "sum"),
        total_impressions=("impressions_90d", "sum")
    )
    .reset_index()
    .rename(columns={"position_bucket": "bucket"})
)
signal2_table["weighted_ctr"] = (
    (signal2_table["total_clicks"] / signal2_table["total_impressions"]) * 100
).round(2)

print("\nSignal 2 Bucket Table (Search Position vs Weighted CTR %):")
print(signal2_table[["bucket", "n", "total_clicks", "total_impressions", "weighted_ctr"]].to_string(index=False))
print("\nVerdict: CONFIRMED — Weighted CTR decreases as search position worsens (0.49% in Top 3 down to 0.04% in Deep positions).")


SIGNAL 1: Staleness / Refresh Signal

Signal 1 Bucket Table (Staleness vs Observed Decline Rate):
     bucket     n  decline_count  decline_rate
  0-30 days 20480          10473         51.14
 31-90 days   175            103         58.86
91-180 days  9171           5604         61.11
  181+ days   174             82         47.13

Verdict: MIXED — 91-180 days shows elevated decline (61.11% vs 51.14% for 0-30 days), providing descriptive support for a refresh flag.

SIGNAL 2: Search Position vs Weighted CTR

Signal 2 Bucket Table (Search Position vs Weighted CTR %):
                     bucket     n  total_clicks  total_impressions  weighted_ctr
                top 3 (1-3)  1141         37042            7560663          0.49
            page 1 (3.1-10) 11842        311928           89361420          0.35
striking distance (10.1-20)  7273         79552           22819980          0.35
        pages 3-5 (20.1-50)  7225         53883           35038692          0.15
                 deep 

## 2. Build the ranked queue

Now I want to turn the baseline idea into a simple review queue that a content reviewer could actually use.

I am keeping the rule intentionally simple:

> **If clicks in the most recent 30 days are lower than clicks in the previous 30 days, flag the page for review.**

In the FlyRank session, this is described as the **April clicks < March clicks** baseline. In this starter dataset, the equivalent fields are `clicks_last_30d` and `clicks_prev_30d`.

A click drop does not automatically mean that a page needs to be refreshed. It only gives us a reason to put the page into the review queue.

After flagging the pages, I rank them using their **most recent 30-day impressions**. This gives higher priority to flagged pages that currently have more search visibility.

The score is therefore deliberately simple:

- **Flagged page:** score = `impressions_last_30d`
- **Not flagged:** score = `0`

Each page also receives one reason code and an action label. This makes the queue easier for a human reviewer to understand.

I also evaluate the baseline using **Precision@K**, with `K = 50`, because the practical question is not just how many pages are flagged, but how useful the first 50 pages are for review.

The baseline is kept separate from the later ML models so that it can act as a fixed point of comparison.

In [22]:
# ML-07 Section 2
# Build and evaluate the transparent baseline action score

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Define the baseline rule
# ---------------------------------------------------------
# Equivalent to:
# "April clicks < March clicks"
#
# In this starter dataset:
# most recent 30 days < previous 30 days

df["baseline_flag"] = (
    df["clicks_last_30d"] < df["clicks_prev_30d"]
)

# ---------------------------------------------------------
# 2. Create the transparent score
# ---------------------------------------------------------
# Rank flagged pages by their recent search visibility.
# Non-flagged pages receive a score of 0.

df["baseline_score"] = np.where(
    df["baseline_flag"],
    df["impressions_last_30d"],
    0
)

# ---------------------------------------------------------
# 3. Add ONE reason code
# ---------------------------------------------------------
df["reason_code"] = np.where(
    df["baseline_flag"],
    "click_decline",
    "no_decline_detected"
)

# ---------------------------------------------------------
# 4. Add an action label
# ---------------------------------------------------------
df["action"] = np.where(
    df["baseline_flag"],
    "refresh_review",
    "monitor"
)

# ---------------------------------------------------------
# 5. Create a deterministic ranked queue
# ---------------------------------------------------------
# Higher score = higher review priority.
# content_id is only used as a tie-breaker.

queue = (
    df.sort_values(
        by=["baseline_score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# 6. Keep the useful columns for the output CSV
# ---------------------------------------------------------
output_cols = [
    "rank",
    "client_id",
    "content_id",
    "clicks_last_30d",
    "clicks_prev_30d",
    "impressions_last_30d",
    "baseline_flag",
    "baseline_score",
    "reason_code",
    "action",
]

baseline_queue = queue[output_cols].copy()

# ---------------------------------------------------------
# 7. Precision@K
# ---------------------------------------------------------
# The starter notebook already defines the observed
# decline proxy as:
# is_declining_label = (trend_direction == "down")
#
# This label is used ONLY for evaluation here.
# It is NOT used to construct the baseline score.

if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (
        df["trend_direction"].eq("down").astype(int)
    )

# Keep the label aligned with the ranked queue
queue["is_declining_label"] = df.loc[
    queue.index, "is_declining_label"
].to_numpy()

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores, kind="stable")
    top_k = order[:k]

    return labels[top_k].mean()

# Calculate Precision@K for several review depths
k_values = [10, 20, 50, 100, 500]

baseline_precision = {}

for k in k_values:
    baseline_precision[f"precision_at_{k}"] = precision_at_k(
        queue["baseline_score"],
        queue["is_declining_label"],
        k=k
    )

# Base rate of the observed decline proxy
base_rate = df["is_declining_label"].mean()

# ---------------------------------------------------------
# 8. Print baseline results
# ---------------------------------------------------------
print("Baseline queue created.")
print(f"Rows: {len(baseline_queue):,}")
print(
    f"Flagged pages: {baseline_queue['baseline_flag'].sum():,}"
    f" ({baseline_queue['baseline_flag'].mean():.2%})"
)
print(
    f"Not flagged: {(~baseline_queue['baseline_flag']).sum():,}"
)
print(f"Observed decline base rate: {base_rate:.2%}")

print("\nPrecision@K:")
for k in k_values:
    print(
        f"Precision@{k}: "
        f"{baseline_precision[f'precision_at_{k}']:.3f}"
    )

# ---------------------------------------------------------
# 9. Save the ranked queue
# ---------------------------------------------------------
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_PATH = OUTPUT_DIR / "baseline_action_score.csv"

baseline_queue.to_csv(
    QUEUE_PATH,
    index=False
)

# ---------------------------------------------------------
# 10. Save the baseline evaluation receipt
# ---------------------------------------------------------
metrics_receipt = {
    "baseline_rule": (
        "clicks_last_30d < clicks_prev_30d"
    ),
    "ranking_field": "impressions_last_30d",
    "review_metric": "Precision@K",
    "primary_k": 50,
    "rows": int(len(df)),
    "flagged_pages": int(baseline_queue["baseline_flag"].sum()),
    "flag_rate": float(baseline_queue["baseline_flag"].mean()),
    "observed_decline_base_rate": float(base_rate),
    **{
        key: float(value)
        for key, value in baseline_precision.items()
    },
}

METRICS_PATH = OUTPUT_DIR / "baseline_metrics.json"

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_receipt, f, indent=2)

# ---------------------------------------------------------
# 11. Display the top 20
# ---------------------------------------------------------
print("\nTop 20 baseline review queue:")
display(baseline_queue.head(20))

print(f"\nQueue written to: {QUEUE_PATH}")
print(f"Metrics written to: {METRICS_PATH}")

Baseline queue created.
Rows: 30,000
Flagged pages: 6,806 (22.69%)
Not flagged: 23,194
Observed decline base rate: 54.21%

Precision@K:
Precision@10: 0.800
Precision@20: 0.650
Precision@50: 0.680
Precision@100: 0.620
Precision@500: 0.548

Top 20 baseline review queue:


,rank,client_id,content_id,clicks_last_30d,clicks_prev_30d,impressions_last_30d,baseline_flag,baseline_score,reason_code,action
0,1,client_19581e27de,content_aaef01a50def,435,461,170559,True,170559,click_decline,refresh_review
1,2,client_6208ef0f77,content_2dba2b1f9536,314,323,139891,True,139891,click_decline,refresh_review
2,3,client_4e07408562,content_5fe46e04994d,220,250,120791,True,120791,click_decline,refresh_review
3,4,client_4e07408562,content_9532f197bbc8,1176,1440,109317,True,109317,click_decline,refresh_review
4,5,client_19581e27de,content_1a9e894be2e2,247,325,107986,True,107986,click_decline,refresh_review
5,6,client_19581e27de,content_36ff89c8214e,35,63,106985,True,106985,click_decline,refresh_review
6,7,client_19581e27de,content_44e481c8f55b,623,803,104458,True,104458,click_decline,refresh_review
7,8,client_19581e27de,content_2c2606c5d176,578,859,104248,True,104248,click_decline,refresh_review
8,9,client_4e07408562,content_8c19996aa890,251,389,89463,True,89463,click_decline,refresh_review
9,10,client_4e07408562,content_4c36c775b818,548,662,83723,True,83723,click_decline,refresh_review



Queue written to: work/outputs/baseline_action_score.csv
Metrics written to: work/outputs/baseline_metrics.json


## 3. Top-20 review

The baseline has now produced a ranked list of pages for human review.

I reviewed the top 20 pages individually instead of treating the baseline score as a final decision.

For each page, I recorded:

1. **Action** — what the baseline recommends.
2. **Reason code** — the signal that caused the page to be flagged.
3. **Why it is here** — the actual click change and search visibility that placed the page high in the queue.
4. **Confidence note** — how convincing the baseline signal looks from the available data.
5. **What would make it wrong** — a realistic reason why the page might not actually need a refresh.

The purpose of this review is to understand the strengths and weaknesses of the simple baseline. A high ranking means that a page should be looked at earlier; it does not mean that the page definitely needs to be changed.

In [26]:
# Section 3: Top-20 human review

# Take the top 20 pages from the final ranked baseline queue
top20 = baseline_queue.head(20).copy()

# Add useful page-level context from the original dataset
context_cols = [
    "content_id",
    "content_type",
    "main_intent",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

top20 = top20.merge(
    df[context_cols],
    on="content_id",
    how="left"
)

# ---------------------------------------------------------
# 1. Why is this page here?
# ---------------------------------------------------------
def get_why_it_is_here(row):
    return (
        f"Clicks dropped from {row['clicks_prev_30d']:,} "
        f"to {row['clicks_last_30d']:,} over the two 30-day periods, "
        f"while the page had {row['impressions_last_30d']:,} recent impressions."
    )


# ---------------------------------------------------------
# 2. Confidence note
# ---------------------------------------------------------
def get_confidence_note(row):
    drop = row["clicks_prev_30d"] - row["clicks_last_30d"]

    if row["impressions_last_30d"] >= 50000 and drop > 0:
        return (
            "Higher confidence in baseline priority because the page has "
            "a clear click decline and substantial search visibility."
        )

    elif drop > 0:
        return (
            "Moderate confidence because clicks declined, but the available "
            "traffic volume is lower than the highest-priority pages."
        )

    return (
        "Low confidence because the baseline signal is weak."
    )


# ---------------------------------------------------------
# 3. What could make the recommendation wrong?
# ---------------------------------------------------------
def get_what_would_make_it_wrong(row):

    pos = row["avg_position"]
    days = row["days_since_last_update"]

    # Recently updated
    if pd.notna(days) and days <= 30:
        return (
            f"Updated recently ({int(days)} days ago); the click decline "
            "could be temporary seasonality or a search-intent change "
            "rather than a content problem."
        )

    # Very strong ranking
    elif pd.notna(pos) and pos > 0 and pos <= 3:
        return (
            f"The page already ranks in the Top 3 (position {pos:.1f}); "
            "the click decline could be related to SERP layout changes "
            "or changing search behavior rather than content quality."
        )

    # Deep ranking
    elif pd.notna(pos) and pos > 20:
        return (
            f"The page has a deeper search position (position {pos:.1f}); "
            "its impressions may come from broad or lower-intent queries, "
            "so high impressions do not necessarily mean a refresh is the "
            "highest-value action."
        )

    # Old content
    elif pd.notna(days) and days >= 90:
        return (
            f"The page has not been updated for {int(days)} days; "
            "the content may be evergreen and intentionally unchanged, "
            "so an unnecessary rewrite could risk existing performance."
        )

    # General case
    else:
        return (
            "The click decline could be caused by seasonality, competition, "
            "or a change in search intent rather than a problem with the content itself."
        )


# Build the review fields
top20["why_it_is_here"] = top20.apply(
    get_why_it_is_here,
    axis=1
)

top20["confidence_note"] = top20.apply(
    get_confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    get_what_would_make_it_wrong,
    axis=1
)


# ---------------------------------------------------------
# 4. Final Top-20 review table
# ---------------------------------------------------------
review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong",
]

review_table = top20[review_cols].copy()

print("=" * 120)
print("TOP-20 BASELINE QUEUE — HUMAN REVIEW")
print("=" * 120)

display(review_table)

print(f"\nTop-20 review rows: {len(review_table)}")

TOP-20 BASELINE QUEUE — HUMAN REVIEW


,rank,content_id,action,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_aaef01a50def,refresh_review,click_decline,Clicks dropped from 461 to 435 over the two 30...,Higher confidence in baseline priority because...,Updated recently (22 days ago); the click decl...
1,2,content_2dba2b1f9536,refresh_review,click_decline,Clicks dropped from 323 to 314 over the two 30...,Higher confidence in baseline priority because...,The page has a deeper search position (positio...
2,3,content_5fe46e04994d,refresh_review,click_decline,Clicks dropped from 250 to 220 over the two 30...,Higher confidence in baseline priority because...,The page has not been updated for 104 days; th...
3,4,content_9532f197bbc8,refresh_review,click_decline,"Clicks dropped from 1,440 to 1,176 over the tw...",Higher confidence in baseline priority because...,The page already ranks in the Top 3 (position ...
4,5,content_1a9e894be2e2,refresh_review,click_decline,Clicks dropped from 325 to 247 over the two 30...,Higher confidence in baseline priority because...,Updated recently (22 days ago); the click decl...
5,6,content_36ff89c8214e,refresh_review,click_decline,Clicks dropped from 63 to 35 over the two 30-d...,Higher confidence in baseline priority because...,The page has not been updated for 104 days; th...
6,7,content_44e481c8f55b,refresh_review,click_decline,Clicks dropped from 803 to 623 over the two 30...,Higher confidence in baseline priority because...,Updated recently (20 days ago); the click decl...
7,8,content_2c2606c5d176,refresh_review,click_decline,Clicks dropped from 859 to 578 over the two 30...,Higher confidence in baseline priority because...,The page has not been updated for 104 days; th...
8,9,content_8c19996aa890,refresh_review,click_decline,Clicks dropped from 389 to 251 over the two 30...,Higher confidence in baseline priority because...,Updated recently (20 days ago); the click decl...
9,10,content_4c36c775b818,refresh_review,click_decline,Clicks dropped from 662 to 548 over the two 30...,Higher confidence in baseline priority because...,Updated recently (20 days ago); the click decl...



Top-20 review rows: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [27]:
# Section 4: Weak picks + leakage check

print("=" * 100)
print("WEAK PICK CHECK")
print("=" * 100)

# ---------------------------------------------------------
# 1. Identify weak picks from the Top-20
# ---------------------------------------------------------
#
# These are not necessarily wrong pages.
# They are pages where the simple baseline has a weaker case
# for prioritising a refresh.

weak_picks = top20.copy()

weak_picks["click_drop"] = (
    weak_picks["clicks_prev_30d"]
    - weak_picks["clicks_last_30d"]
)

weak_picks["click_drop_pct"] = (
    weak_picks["click_drop"]
    / weak_picks["clicks_prev_30d"].replace(0, np.nan)
) * 100


# A weak pick can occur when:
# - the page was updated recently
# - the absolute click decline is very small
# - the page has a deep search position

weak_candidates = weak_picks[
    (
        (weak_picks["days_since_last_update"] <= 30)
        |
        (weak_picks["click_drop"] <= 10)
        |
        (weak_picks["avg_position"] > 20)
    )
].copy()

# Keep a few examples for inspection
weak_candidates = weak_candidates.sort_values(
    by="rank"
).head(5)

print(f"Weak-pick candidates found: {len(weak_candidates)}")

display(
    weak_candidates[
        [
            "rank",
            "content_id",
            "clicks_last_30d",
            "clicks_prev_30d",
            "click_drop",
            "click_drop_pct",
            "impressions_last_30d",
            "days_since_last_update",
            "avg_position",
            "action",
            "reason_code",
        ]
    ]
)


# ---------------------------------------------------------
# 2. Leakage check
# ---------------------------------------------------------
print("\n" + "=" * 100)
print("BASELINE LEAKAGE CHECK")
print("=" * 100)

# Fields directly used to construct the baseline score
baseline_inputs = [
    "clicks_last_30d",
    "clicks_prev_30d",
    "impressions_last_30d",
]

# Fields that must NOT be used to construct the score
forbidden_inputs = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
]

print("\nBaseline score inputs:")
for col in baseline_inputs:
    print(f"  - {col}")

print("\nChecking forbidden fields:")

leaked_fields = [
    col for col in forbidden_inputs
    if col in baseline_inputs
]

assert len(leaked_fields) == 0, (
    f"Leakage detected in baseline inputs: {leaked_fields}"
)

print("  ✓ No target-derived fields or IDs are used in the baseline score.")

# ---------------------------------------------------------
# 3. Confirm the score is exactly the intended rule
# ---------------------------------------------------------
expected_score = np.where(
    baseline_queue["baseline_flag"],
    baseline_queue["impressions_last_30d"],
    0
)

assert np.array_equal(
    baseline_queue["baseline_score"].to_numpy(),
    expected_score
)

print("  ✓ Baseline score matches the documented rule.")

# ---------------------------------------------------------
# 4. Confirm the flag itself uses only the two click windows
# ---------------------------------------------------------
expected_flag = (
    baseline_queue["clicks_last_30d"]
    < baseline_queue["clicks_prev_30d"]
)

assert np.array_equal(
    baseline_queue["baseline_flag"].to_numpy(),
    expected_flag.to_numpy()
)

print("  ✓ Baseline flag matches clicks_last_30d < clicks_prev_30d.")

print("\nLeakage check passed.")

WEAK PICK CHECK
Weak-pick candidates found: 5


,rank,content_id,clicks_last_30d,clicks_prev_30d,click_drop,click_drop_pct,impressions_last_30d,days_since_last_update,avg_position,action,reason_code
0,1,content_aaef01a50def,435,461,26,5.639913,170559,22,5.4,refresh_review,click_decline
1,2,content_2dba2b1f9536,314,323,9,2.786378,139891,104,27.9,refresh_review,click_decline
4,5,content_1a9e894be2e2,247,325,78,24.000000,107986,22,4.0,refresh_review,click_decline
6,7,content_44e481c8f55b,623,803,180,22.415940,104458,20,1.4,refresh_review,click_decline
8,9,content_8c19996aa890,251,389,138,35.475578,89463,20,2.5,refresh_review,click_decline



BASELINE LEAKAGE CHECK

Baseline score inputs:
  - clicks_last_30d
  - clicks_prev_30d
  - impressions_last_30d

Checking forbidden fields:
  ✓ No target-derived fields or IDs are used in the baseline score.
  ✓ Baseline score matches the documented rule.
  ✓ Baseline flag matches clicks_last_30d < clicks_prev_30d.

Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.